In [3]:
# ==============================================================================
# Premier League 2024/25 — Interactive Position Race Visualiser
# Leider Ivan Páez Páez — StatsBomb Data
# ==============================================================================


# ── CELL 1 ── Libraries ───────────────────────────────────────────────────────
import re
import base64
import requests
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go
from google.colab import drive

drive.mount('/content/drive')


# ── CELL 2 ── Configuration ───────────────────────────────────────────────────

PROJECT_FOLDER = "/content/drive/MyDrive/Premier League 2024 2025/Matches"
LOGOS_PATH     = "/content/drive/MyDrive/Premier League 2024 2025/Premier League Logos.csv"
OUTPUT_HTML    = "/content/drive/MyDrive/Premier League 2024 2025/pl_position_race.html"

# Official club colours (adjusted for dark background contrast)
TEAM_COLORS = {
    "Arsenal":                    "#EF0107",
    "Aston Villa":                "#95BFE5",
    "AFC Bournemouth":            "#DA291C",
    "Brentford":                  "#FFD700",
    "Brighton & Hove Albion":     "#0057B8",
    "Chelsea":                    "#034694",
    "Crystal Palace":             "#C4122E",
    "Everton":                    "#003399",
    "Fulham":                     "#FFFFFF",
    "Ipswich Town":               "#4287C8",
    "Leicester City":             "#003090",
    "Liverpool":                  "#C8102E",
    "Manchester City":            "#6CABDD",
    "Manchester United":          "#F5A623",
    "Newcastle United":           "#AAAAAA",
    "Nottingham Forest":          "#FF6B35",
    "Southampton":                "#FFC0CB",
    "Tottenham Hotspur":          "#8EB4E3",
    "West Ham United":            "#7A263A",
    "Wolverhampton Wanderers":    "#FDB913",
}

# Logo CSV name → StatsBomb team name
LOGO_NAME_MAP = {
    "Bournemouth":              "AFC Bournemouth",
    "Brighton and Hove Albion": "Brighton & Hove Albion",
}

# ── Figure dimensions ─────────────────────────────────────────────────────────
FIGURE_WIDTH  = 1200
FIGURE_HEIGHT = 860
MARGIN        = dict(l=55, r=60, t=90, b=115)

X_DATA_MIN, X_DATA_MAX_EXTRA = 0.5, 3

# Default logo size (data units) — same as v3
LOGO_SIZEX_DEFAULT = 1.6
LOGO_SIZEY_DEFAULT = 0.82

# Per-team overrides for logos with a different SVG aspect ratio
LOGO_SIZE_OVERRIDES = {
    "Tottenham Hotspur": (0.5, 0.72),   # SVG has tighter viewBox → scale down
}


# ── CELL 3 ── Load logos and embed as base64 data URIs ───────────────────────

logos_df = pd.read_csv(LOGOS_PATH, encoding="utf-8-sig")
logos_df.columns      = logos_df.columns.str.strip()
logos_df["Team"]      = logos_df["Team"].str.strip()
logos_df["Logo link"] = logos_df["Logo link"].str.strip()
logos_df["Team"]      = logos_df["Team"].replace(LOGO_NAME_MAP)

LOGO_MAP = dict(zip(logos_df["Team"], logos_df["Logo link"]))
print(f"✓ {len(LOGO_MAP)} team logos loaded")


def logo_to_datauri(url: str) -> str:
    """Downloads a logo and returns it as an embedded base64 data URI.
    Avoids CORS issues when Plotly tries to load external SVG URLs."""
    try:
        r = requests.get(url, timeout=8)
        r.raise_for_status()
        mime = "image/svg+xml" if url.lower().endswith(".svg") else "image/png"
        b64  = base64.b64encode(r.content).decode()
        return f"data:{mime};base64,{b64}"
    except Exception as e:
        print(f"  ⚠ Could not fetch {url}: {e}")
        return url


print("Embedding logos as base64...")
LOGO_DATAURI = {}
for team, url in LOGO_MAP.items():
    LOGO_DATAURI[team] = logo_to_datauri(url)
    print(f"  ✓ {team}")

print(f"\n✓ {len(LOGO_DATAURI)} logos ready")


# ── CELL 4 ── Match processing function ───────────────────────────────────────

def match_team_name(raw: str, candidates: list) -> str:
    """Maps a filename-derived name to the real team name in the data."""
    first_word = raw.lower().split()[0]
    for c in candidates:
        if c.lower().split()[0] == first_word:
            return c
    for c in candidates:
        if first_word in c.lower():
            return c
    return candidates[0]


def process_match(file_path: str) -> dict:
    base  = Path(file_path).stem.replace("_", " ")
    parts = re.split(r" [Vv] ", base, maxsplit=1)
    h_raw = parts[0].strip()

    df   = pd.read_csv(file_path, low_memory=False)
    m_id = int(df["match_id"].iloc[0])

    actual_teams = df["team_name"].dropna().unique().tolist()
    h_team = match_team_name(h_raw, actual_teams)
    a_team = next((t for t in actual_teams if t != h_team), actual_teams[-1])

    normal_goals = (
        df[(df["event_type_name"] == "Shot") & (df["outcome_name"] == "Goal")]
        .drop_duplicates("id")
        .groupby("team_name").size().rename("goals")
    )
    own_goals = (
        df[df["event_type_name"] == "Own Goal For"]
        .drop_duplicates("id")
        .groupby("team_name").size().rename("goals")
    )

    total = pd.concat([normal_goals, own_goals]).groupby("team_name").sum()

    h_g   = int(total.get(h_team, 0))
    a_g   = int(total.get(a_team, 0))
    h_pts = 3 if h_g > a_g else (1 if h_g == a_g else 0)
    a_pts = 3 if a_g > h_g else (1 if a_g == h_g else 0)

    return dict(
        match_id=m_id,
        home_team=h_team, home_goals=h_g,
        away_team=a_team, away_goals=a_g,
        home_points=h_pts, away_points=a_pts,
    )


# ── CELL 5 ── Batch processing ────────────────────────────────────────────────

file_list = sorted(Path(PROJECT_FOLDER).glob("*.csv"))
print(f"✓ {len(file_list)} CSV files found")

results_list = []
for i, f in enumerate(file_list, 1):
    try:
        results_list.append(process_match(str(f)))
    except Exception as e:
        print(f"  ✗ Error in {f.name}: {e}")
    if i % 50 == 0:
        print(f"  → {i}/{len(file_list)} matches processed")

results = (
    pd.DataFrame(results_list)
    .sort_values("match_id")
    .reset_index(drop=True)
)

print(f"\n✓ {len(results)} matches processed")
print(f"  Total home goals : {results['home_goals'].sum()}")
print(f"  Total away goals : {results['away_goals'].sum()}")


# ── CELL 6 ── Assign matchweeks (10 matches per round) ───────────────────────

results["match_num"] = results.index + 1
results["matchweek"] = ((results["match_num"] - 1) // 10) + 1
MAX_MW = int(results["matchweek"].max())
print(f"✓ {MAX_MW} matchweeks detected")


# ── CELL 7 ── Build cumulative standings per matchweek ────────────────────────

all_teams = sorted(set(results["home_team"]) | set(results["away_team"]))
records   = []

for mw in range(1, MAX_MW + 1):
    sub = results[results["matchweek"] <= mw]

    home = sub[["home_team","home_goals","away_goals","home_points"]].rename(
        columns={"home_team":"team","home_goals":"gf","away_goals":"ga","home_points":"pts"})
    away = sub[["away_team","away_goals","home_goals","away_points"]].rename(
        columns={"away_team":"team","away_goals":"gf","home_goals":"ga","away_points":"pts"})

    standings = (
        pd.concat([home, away])
        .groupby("team")
        .agg(MP=("pts","count"), Pts=("pts","sum"), GF=("gf","sum"), GA=("ga","sum"))
        .assign(GD=lambda x: x.GF - x.GA, W=0, D=0, L=0)
        .reset_index()
        .sort_values(["Pts","GD","GF"], ascending=[False,False,False])
        .reset_index(drop=True)
    )

    for _, row in sub.iterrows():
        for team, pts in [(row.home_team, row.home_points), (row.away_team, row.away_points)]:
            idx = standings[standings["team"] == team].index
            if len(idx):
                i = idx[0]
                if   pts == 3: standings.at[i, "W"] += 1
                elif pts == 1: standings.at[i, "D"] += 1
                else:          standings.at[i, "L"] += 1

    standings["Pos"] = standings.index + 1
    standings["MW"]  = mw
    records.append(standings)

standings_all = pd.concat(records, ignore_index=True)
print("✓ Cumulative standings table built")


# ── CELL 8 ── Animated Bump Chart with Embedded Team Logos ───────────────────
#
# Palette: Gol Caracol official — #016AF2 (blue), #00249C (dark blue),
#          #D41810 (red), #FFFFFF (white).
# Features: dark/light theme toggle, zone legend below x-axis.
# ─────────────────────────────────────────────────────────────────────────────

# ── Dark theme (default) ──────────────────────────────────────────────────────
DARK = dict(
    BG        = "#020a1e",
    PANEL     = "#06122e",
    TEXT_PRI  = "#FFFFFF",
    TEXT_SEC  = "#6a85b5",
    GRID      = "rgba(1,106,242,0.10)",
    BORDER    = "rgba(1,106,242,0.30)",
    HOVER_BG  = "#071530",
    LEGEND_BG = "rgba(2,10,30,0.80)",
    BTN_BG    = "rgba(1,106,242,0.15)",
    SLD_BG    = "rgba(1,106,242,0.08)",
)

# ── Light theme ───────────────────────────────────────────────────────────────
LIGHT = dict(
    BG        = "#FFFFFF",
    PANEL     = "#F0F5FF",
    TEXT_PRI  = "#00249C",
    TEXT_SEC  = "#5070A0",
    GRID      = "rgba(0,36,156,0.08)",
    BORDER    = "rgba(0,36,156,0.22)",
    HOVER_BG  = "#E8EFFF",
    LEGEND_BG = "rgba(240,245,255,0.92)",
    BTN_BG    = "rgba(0,36,156,0.10)",
    SLD_BG    = "rgba(0,36,156,0.06)",
)

THEME = DARK   # default — JS toggle switches between both

BLUE   = "#016AF2"    # Gol Caracol primary blue
DBLUE  = "#00249C"    # Gol Caracol dark blue
RED    = "#D41810"    # Gol Caracol red

FONT_TITLE = "Impact, Arial Black, sans-serif"
FONT_BODY  = "DM Sans, Helvetica Neue, sans-serif"

# ── Gol Caracol logo (SVG built from brand colors — no external request) ──────
import base64 as _b64
_GOL_SVG = f"""<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 320 110">
  <defs>
    <linearGradient id="bg" x1="0%" y1="0%" x2="100%" y2="0%">
      <stop offset="0%"   stop-color="{DBLUE}"/>
      <stop offset="70%"  stop-color="#010e3d"/>
      <stop offset="100%" stop-color="{RED}"/>
    </linearGradient>
    <linearGradient id="pill" x1="0%" y1="0%" x2="100%" y2="100%">
      <stop offset="0%"   stop-color="{BLUE}"/>
      <stop offset="100%" stop-color="{DBLUE}"/>
    </linearGradient>
  </defs>
  <!-- Background -->
  <rect width="320" height="110" rx="14" fill="url(#bg)"/>
  <!-- Blue pill -->
  <rect x="10" y="10" width="220" height="90" rx="10" fill="url(#pill)"/>
  <!-- GOL wordmark -->
  <text x="118" y="80" font-family="Impact,Arial Black,sans-serif"
        font-size="72" font-weight="900" fill="white"
        text-anchor="middle" letter-spacing="3">GOL</text>
  <!-- Caracol snail icon (right side) -->
  <g transform="translate(272,55)">
    <circle r="30" fill="none" stroke="white" stroke-width="4"/>
    <circle r="19" fill="none" stroke="white" stroke-width="3.5"/>
    <circle r="8"  fill="none" stroke="white" stroke-width="3"/>
    <circle r="3"  fill="white"/>
    <!-- antenna -->
    <line x1="0" y1="-30" x2="0" y2="-42" stroke="white" stroke-width="3"
          stroke-linecap="round"/>
    <circle cx="0" cy="-46" r="4" fill="{RED}"/>
  </g>
</svg>"""
GOL_LOGO_URI = "data:image/svg+xml;base64," + _b64.b64encode(_GOL_SVG.encode()).decode()

# ── Qualification zones ────────────────────────────────────────────────────────
# solid_color is used for the legend swatch (opaque); color for the band fill.
ZONES = [
    {"y0": 0.5,  "y1": 4.5,  "solid": "#016AF2", "color": "rgba(1,106,242,0.09)",  "label": "UEFA Champions League"},
    {"y0": 4.5,  "y1": 5.5,  "solid": "#FF8C00", "color": "rgba(255,140,0,0.08)",  "label": "UEFA Europa League"},
    {"y0": 5.5,  "y1": 6.5,  "solid": "#00A878", "color": "rgba(0,168,120,0.07)",  "label": "UEFA Conference League"},
    {"y0": 17.5, "y1": 20.5, "solid": "#D41810", "color": "rgba(212,24,16,0.10)",  "label": "Relegation"},
]

# Zone legend — vertical list on the RIGHT, below the clubs legend box.
# With 20 teams the clubs legend ends around y≈0.42 in paper coords.
# Each zone gets a colored swatch rect + label annotation.
ZONE_LEGEND_TOP  = 0.18   # paper y where zone legend header sits
ZONE_LEGEND_STEP = 0.058  # vertical gap between items

ZONE_SHAPES_LEGEND = [
    dict(type="rect", xref="paper", yref="paper",
         x0=1.010, x1=1.030,
         y0=ZONE_LEGEND_TOP - 0.028 - i*ZONE_LEGEND_STEP,
         y1=ZONE_LEGEND_TOP + 0.008 - i*ZONE_LEGEND_STEP,
         fillcolor=z["solid"], line_width=0, layer="above")
    for i, z in enumerate(ZONES)
]
ZONE_ANNOTATIONS_LEGEND = [
    # Section header
    dict(x=1.010, y=ZONE_LEGEND_TOP + 0.030,
         xref="paper", yref="paper",
         xanchor="left", yanchor="middle",
         text="COMPETITION ZONES",
         font=dict(size=8, color=THEME["TEXT_SEC"],
                   family=FONT_BODY),
         showarrow=False),
] + [
    # One label per zone
    dict(x=1.040, y=ZONE_LEGEND_TOP - 0.010 - i*ZONE_LEGEND_STEP,
         xref="paper", yref="paper",
         xanchor="left", yanchor="middle",
         text=z["label"],
         font=dict(size=8.5, color=THEME["TEXT_SEC"], family=FONT_BODY),
         showarrow=False)
    for i, z in enumerate(ZONES)
]


def build_hover(row) -> str:
    return (
        f"<b style='color:{BLUE}'>{row.team}</b><br>"
        f"<span style='color:{THEME['TEXT_SEC']}'>Matchweek {row.MW}</span><br>"
        f"Position: <b>{row.Pos}</b><br>"
        f"Points: <b>{row.Pts}</b> &nbsp;·&nbsp; "
        f"W-D-L: {row.W}-{row.D}-{row.L}<br>"
        f"GF: {row.GF} &nbsp; GA: {row.GA} &nbsp; GD: {row.GD:+d}"
    )


def make_logo_image(uri: str, mw: int, pos: float, team: str = "") -> dict:
    sizex, sizey = LOGO_SIZE_OVERRIDES.get(team, (LOGO_SIZEX_DEFAULT, LOGO_SIZEY_DEFAULT))
    return dict(
        source=uri, xref="x", yref="y",
        x=mw + 0.5, y=pos,
        sizex=sizex, sizey=sizey,
        xanchor="center", yanchor="middle",
        layer="above",
    )



# ── Initial traces (Matchweek 1) ──────────────────────────────────────────────
initial_data = []
for team in all_teams:
    td    = standings_all[(standings_all["team"] == team) & (standings_all["MW"] == 1)]
    color = TEAM_COLORS.get(team, "#aaaaaa")
    initial_data.append(go.Scatter(
        x    = td["MW"],
        y    = td["Pos"],
        mode = "lines+markers",
        name = team,
        line = dict(color=color, width=2.2, shape="spline", smoothing=0.9),
        marker = dict(size=7, color=color,
                      line=dict(width=1.2, color=THEME["PANEL"]),
                      opacity=0.92),
        hovertemplate = "%{customdata}<extra></extra>",
        customdata    = [build_hover(r) for r in td.itertuples()],
        showlegend    = True,
    ))

initial_images = []
for team in all_teams:
    td  = standings_all[(standings_all["team"] == team) & (standings_all["MW"] == 1)]
    uri = LOGO_DATAURI.get(team)
    if uri and not td.empty:
        initial_images.append(make_logo_image(uri, 1, td.iloc[-1]["Pos"], team))


# ── Animation frames ──────────────────────────────────────────────────────────
frames = []
for mw in range(1, MAX_MW + 1):
    frame_data   = []
    frame_images = []

    for team in all_teams:
        td    = standings_all[(standings_all["team"] == team) & (standings_all["MW"] <= mw)].sort_values("MW")
        color = TEAM_COLORS.get(team, "#aaaaaa")
        sizes = [5] * max(0, len(td) - 1) + [10]

        frame_data.append(go.Scatter(
            x    = td["MW"],
            y    = td["Pos"],
            mode = "lines+markers",
            name = team,
            line = dict(color=color, width=2.2, shape="spline", smoothing=0.9),
            marker = dict(size=sizes, color=color,
                          line=dict(width=1.2, color=THEME["PANEL"]),
                          opacity=0.92),
            hovertemplate = "%{customdata}<extra></extra>",
            customdata    = [build_hover(r) for r in td.itertuples()],
        ))

        uri = LOGO_DATAURI.get(team)
        if uri and not td.empty:
            frame_images.append(make_logo_image(uri, mw, td.iloc[-1]["Pos"], team))

    frames.append(go.Frame(
        data   = frame_data,
        layout = go.Layout(
            annotations = ZONE_ANNOTATIONS_LEGEND,
            images      = frame_images,
        ),
        name = str(mw),
    ))


# ── Slider steps ──────────────────────────────────────────────────────────────
FRAME_DURATION_MS = 450
TRANSITION_MS     = 380

slider_steps = [
    dict(
        args   = [[str(mw)], {"frame":      {"duration": FRAME_DURATION_MS, "redraw": True},
                               "mode":       "immediate",
                               "transition": {"duration": TRANSITION_MS, "easing": "cubic-in-out"}}],
        label  = str(mw),
        method = "animate",
    )
    for mw in range(1, MAX_MW + 1)
]


# ── Layout ────────────────────────────────────────────────────────────────────
layout = go.Layout(
    title = dict(
        text = (
            "<span style='letter-spacing:0.10em'>PREMIER LEAGUE 2024 · 25</span>"
            f"<br><sup style='font-family:{FONT_BODY};letter-spacing:0.18em;"
            f"color:{THEME['TEXT_SEC']}'>TITLE RACE — MATCHWEEK BY MATCHWEEK</sup>"
        ),
        font    = dict(size=26, color=THEME["TEXT_PRI"], family=FONT_TITLE),
        x=0.5, xanchor="center", pad=dict(t=14),
    ),
    width         = FIGURE_WIDTH,
    height        = FIGURE_HEIGHT,
    paper_bgcolor = THEME["BG"],
    plot_bgcolor  = THEME["PANEL"],
    font          = dict(family=FONT_BODY, color=THEME["TEXT_SEC"], size=11),
    margin        = dict(l=55, r=60, t=90, b=115),

    xaxis = dict(
        title     = dict(text="MATCHWEEK",
                         font=dict(size=10, color=THEME["TEXT_SEC"], family=FONT_BODY),
                         standoff=14),
        range     = [X_DATA_MIN, MAX_MW + X_DATA_MAX_EXTRA],
        tickvals  = list(range(1, MAX_MW + 1)),
        tickfont  = dict(size=9.5, color=THEME["TEXT_SEC"], family=FONT_BODY),
        gridcolor = THEME["GRID"],  showgrid=True,
        zeroline  = False, fixedrange=False,
        linecolor = THEME["BORDER"], linewidth=1, showline=True,
    ),
    yaxis = dict(
        title     = dict(text="POSITION",
                         font=dict(size=10, color=THEME["TEXT_SEC"], family=FONT_BODY),
                         standoff=10),
        autorange = "reversed",
        range     = [20.7, 0.3],
        tickvals  = list(range(1, 21)),
        tickfont  = dict(size=9.5, color=THEME["TEXT_SEC"], family=FONT_BODY),
        gridcolor = THEME["GRID"],  showgrid=True,
        zeroline  = False, fixedrange=True,
        linecolor = THEME["BORDER"], linewidth=1, showline=True,
    ),

    # Zone bands (plot area fill) + thin left-edge accent lines
    shapes = [
        dict(type="rect", xref="paper", yref="y",
             x0=0, x1=1, y0=z["y0"], y1=z["y1"],
             fillcolor=z["color"], line_width=0, layer="below")
        for z in ZONES
    ] + [
        dict(type="line", xref="paper", yref="y",
             x0=0, x1=0, y0=z["y0"], y1=z["y1"],
             line=dict(color=z["solid"], width=2.5, dash="solid"), layer="above")
        for z in ZONES
    ] + ZONE_SHAPES_LEGEND,

    annotations = ZONE_ANNOTATIONS_LEGEND,
    images      = initial_images,

    hoverlabel = dict(
        bgcolor     = THEME["HOVER_BG"],
        bordercolor = BLUE,
        font        = dict(family=FONT_BODY, size=12, color=THEME["TEXT_PRI"]),
    ),

    legend = dict(
        bgcolor     = THEME["LEGEND_BG"],
        bordercolor = THEME["BORDER"],
        borderwidth = 1,
        font        = dict(size=9, family=FONT_BODY, color=THEME["TEXT_SEC"]),
        title       = dict(text="CLUBS",
                           font=dict(size=8.5, color=THEME["TEXT_SEC"], family=FONT_BODY)),
        x=1.01, y=1, xanchor="left", yanchor="top",
        tracegroupgap=0,
    ),

    updatemenus = [dict(
        type="buttons", showactive=False,
        x=0.0, y=-0.13, xanchor="left", yanchor="top",
        buttons=[
            dict(label="▶  PLAY", method="animate",
                 args=[None, {"frame":       {"duration": FRAME_DURATION_MS, "redraw": True},
                              "fromcurrent": True,
                              "transition":  {"duration": TRANSITION_MS, "easing": "cubic-in-out"}}]),
            dict(label="⏸  PAUSE", method="animate",
                 args=[[None], {"frame": {"duration": 0, "redraw": False},
                                "mode": "immediate", "transition": {"duration": 0}}]),
        ],
        font=dict(size=11, family=FONT_BODY, color=THEME["TEXT_PRI"]),
        bgcolor=THEME["BTN_BG"], bordercolor=BLUE, borderwidth=1,
        pad=dict(l=12, r=12, t=7, b=7),
    )],

    sliders=[dict(
        active=0,
        currentvalue=dict(
            prefix="MATCHWEEK  ",
            font=dict(size=13, color=BLUE, family=FONT_BODY),
            visible=True, xanchor="center",
        ),
        pad=dict(t=55, b=10),
        x=0.13, len=0.77,
        bgcolor=THEME["SLD_BG"], bordercolor=THEME["BORDER"],
        tickcolor=THEME["BORDER"],
        font=dict(color=THEME["TEXT_SEC"], size=8.5, family=FONT_BODY),
        steps=slider_steps,
    )],
)


# ── Assemble & export ─────────────────────────────────────────────────────────
fig = go.Figure(data=initial_data, layout=layout, frames=frames)
fig.show()
fig.write_html(OUTPUT_HTML, include_plotlyjs="cdn", full_html=True)

# ── Inject fonts + CSS transitions + dark/light toggle button ─────────────────
INJECT = f"""
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=DM+Sans:wght@300;400;500&display=swap" rel="stylesheet">
<style>
  body {{ background:{DARK['BG']}; margin:0; transition: background 0.4s; }}

  /* Smooth logo glide between frames */
  .main-svg image {{
    transition: transform {TRANSITION_MS}ms cubic-bezier(0.4,0,0.2,1),
                x         {TRANSITION_MS}ms cubic-bezier(0.4,0,0.2,1),
                y         {TRANSITION_MS}ms cubic-bezier(0.4,0,0.2,1);
  }}

  /* Plot panel border */
  .main-svg .bg {{ stroke:rgba(1,106,242,0.25) !important; stroke-width:1px !important; }}

  /* ── Theme toggle button ── */
  #theme-btn {{
    position: fixed;
    top: 14px; right: 18px;
    z-index: 9999;
    background: rgba(1,106,242,0.18);
    color: #FFFFFF;
    border: 1.5px solid #016AF2;
    border-radius: 6px;
    padding: 6px 16px;
    font-family: 'DM Sans', sans-serif;
    font-size: 12px;
    letter-spacing: 0.08em;
    cursor: pointer;
    transition: background 0.25s, color 0.25s;
  }}
  #theme-btn:hover {{ background: rgba(1,106,242,0.35); }}
  body.light #theme-btn {{
    background: rgba(0,36,156,0.10);
    color: #00249C;
    border-color: #00249C;
  }}
  body.light {{ background: #FFFFFF; }}
  body.light .main-svg .bg {{
    stroke: rgba(0,36,156,0.20) !important;
  }}
</style>
<button id="theme-btn" onclick="toggleTheme()">☀ LIGHT MODE</button>
<script>
var isDark = true;

var DARK_THEME  = {{
  paper_bgcolor: '{DARK["BG"]}',
  plot_bgcolor:  '{DARK["PANEL"]}',
  'font.color':              '{DARK["TEXT_SEC"]}',
  'title.font.color':        '{DARK["TEXT_PRI"]}',
  'xaxis.gridcolor':         '{DARK["GRID"]}',
  'xaxis.linecolor':         '{DARK["BORDER"]}',
  'xaxis.tickfont.color':    '{DARK["TEXT_SEC"]}',
  'xaxis.title.font.color':  '{DARK["TEXT_SEC"]}',
  'yaxis.gridcolor':         '{DARK["GRID"]}',
  'yaxis.linecolor':         '{DARK["BORDER"]}',
  'yaxis.tickfont.color':    '{DARK["TEXT_SEC"]}',
  'yaxis.title.font.color':  '{DARK["TEXT_SEC"]}',
  'legend.bgcolor':          '{DARK["LEGEND_BG"]}',
  'legend.bordercolor':      '{DARK["BORDER"]}',
  'legend.font.color':       '{DARK["TEXT_SEC"]}',
}};

var LIGHT_THEME = {{
  paper_bgcolor: '{LIGHT["BG"]}',
  plot_bgcolor:  '{LIGHT["PANEL"]}',
  'font.color':              '{LIGHT["TEXT_SEC"]}',
  'title.font.color':        '{LIGHT["TEXT_PRI"]}',
  'xaxis.gridcolor':         '{LIGHT["GRID"]}',
  'xaxis.linecolor':         '{LIGHT["BORDER"]}',
  'xaxis.tickfont.color':    '{LIGHT["TEXT_SEC"]}',
  'xaxis.title.font.color':  '{LIGHT["TEXT_SEC"]}',
  'yaxis.gridcolor':         '{LIGHT["GRID"]}',
  'yaxis.linecolor':         '{LIGHT["BORDER"]}',
  'yaxis.tickfont.color':    '{LIGHT["TEXT_SEC"]}',
  'yaxis.title.font.color':  '{LIGHT["TEXT_SEC"]}',
  'legend.bgcolor':          '{LIGHT["LEGEND_BG"]}',
  'legend.bordercolor':      '{LIGHT["BORDER"]}',
  'legend.font.color':       '{LIGHT["TEXT_SEC"]}',
}};

function toggleTheme() {{
  var gd  = document.querySelector('.plotly-graph-div');
  var btn = document.getElementById('theme-btn');
  if (isDark) {{
    Plotly.relayout(gd, LIGHT_THEME);
    document.body.classList.add('light');
    btn.textContent = '🌙 DARK MODE';
  }} else {{
    Plotly.relayout(gd, DARK_THEME);
    document.body.classList.remove('light');
    btn.textContent = '☀ LIGHT MODE';
  }}
  isDark = !isDark;
}}
</script>
"""

with open(OUTPUT_HTML, "r", encoding="utf-8") as f:
    html = f.read()
html = html.replace("</head>", INJECT + "\n</head>")
with open(OUTPUT_HTML, "w", encoding="utf-8") as f:
    f.write(html)

print(f"\n✓ Visualisation saved to:\n  {OUTPUT_HTML}")
print("  (CSS transitions + dark/light toggle injected)")


# ── CELL 9 (optional) ── Final standings table ────────────────────────────────

final_standings = (
    standings_all[standings_all["MW"] == MAX_MW]
    [["Pos","team","MP","W","D","L","GF","GA","GD","Pts"]]
    .rename(columns={"team": "Team"})
    .reset_index(drop=True)
)

print("\n📊 FINAL STANDINGS — Premier League 2024/25")
print("=" * 65)
print(final_standings.to_string(index=False))

Output hidden; open in https://colab.research.google.com to view.

In [28]:
import os

GITHUB_USER  = "LeiderIP"
GITHUB_TOKEN = "YOUR_GITHUB_TOKEN"
GITHUB_REPO  = "Premier-League-2024-25-Title-race"

os.chdir(f"/content/{GITHUB_REPO}")

# Descarga el README desde la URL raw del archivo que te di antes
!wget -q -O README.md "https://raw.githubusercontent.com/LeiderIP/Premier-League-2024-25-Title-race/main/README.md"

# Escribe el README línea por línea evitando el conflicto con las triple comillas
lines = [
    "# Premier League 2024/25 — Title Race Visualiser\n",
    "\n",
    "An interactive animated bump chart tracking the position of all 20 Premier League clubs across every matchweek of the 2024/25 season, built entirely from raw StatsBomb event data.\n",
    "\n",
    "![Python](https://img.shields.io/badge/Python-3.10+-blue?style=flat-square&logo=python)\n",
    "![Plotly](https://img.shields.io/badge/Plotly-5.x-informational?style=flat-square&logo=plotly)\n",
    "![Data](https://img.shields.io/badge/Data-StatsBomb-red?style=flat-square)\n",
    "![Colab](https://img.shields.io/badge/Platform-Google%20Colab-orange?style=flat-square&logo=google-colab)\n",
    "\n",
    "---\n",
    "\n",
    "## 🔴 Live Demo\n",
    "\n",
    "**[▶ Open Interactive Visualisation](https://leiderip.github.io/Premier-League-2024-25-Title-race/pl_position_race.html)**\n",
    "\n",
    "---\n",
    "\n",
    "## Overview\n",
    "\n",
    "This project processes **380 match CSV files** from StatsBomb's event-level dataset and builds a fully interactive league standings animation — from raw events to polished visualisation — with no pre-aggregated data.\n",
    "\n",
    "The chart shows how each club's league position evolved week by week, revealing the full narrative of the title race, the relegation battle, and the European qualification fight in a single interactive view.\n",
    "\n",
    "---\n",
    "\n",
    "## Features\n",
    "\n",
    "- **Animated bump chart** — 38-matchweek position race with Play / Pause control and a matchweek slider\n",
    "- **Team badges** — Premier League SVG crests embedded as base64 (no external requests, CORS-free)\n",
    "- **Dark / Light theme toggle** — switch between dark navy and clean white with a single click\n",
    "- **Zone bands** — visual markers for Champions League, Europa League, Conference League, and Relegation zones\n",
    "- **Rich tooltips** — hover any point to see Position, Points, W-D-L, GF, GA, and GD for that matchweek\n",
    "- **Fully self-contained HTML** — the exported file runs offline in any modern browser\n",
    "\n",
    "---\n",
    "\n",
    "## Data Pipeline\n",
    "\n",
    "| Step | Description |\n",
    "|---|---|\n",
    "| Goal extraction | Shot + Goal outcome, deduplicated on event id |\n",
    "| Own goal attribution | 'Own Goal For' events credited to beneficiary team |\n",
    "| Match result calculation | W/D/L and points per match |\n",
    "| Cumulative standings | Built for every matchweek (Pts → GD → GF tiebreak) |\n",
    "| Plotly animated figure | 38 frames, one per matchweek |\n",
    "| HTML export + CSS inject | Smooth logo transitions, font loading, theme toggle |\n",
    "\n",
    "---\n",
    "\n",
    "## Key Technical Decisions\n",
    "\n",
    "| Challenge | Solution |\n",
    "|---|---|\n",
    "| Freeze-frame rows duplicating goal events | drop_duplicates('id') before counting |\n",
    "| Team name mismatch between filename and data | First-word matching against team_name column |\n",
    "| Logo CORS block in browser | Downloaded and embedded as base64 data URIs |\n",
    "| Logos snapping between frames | CSS transition injected into exported HTML |\n",
    "| Tottenham SVG tighter viewBox | Per-team sizex/sizey override dictionary |\n",
    "| Zone labels overlapping team logos | Moved to paper-coordinate legend panel |\n",
    "\n",
    "---\n",
    "\n",
    "## Stack\n",
    "\n",
    "| Tool | Purpose |\n",
    "|---|---|\n",
    "| pandas | Event data loading, goal aggregation, standings computation |\n",
    "| plotly | Animated interactive chart, frame-by-frame layout |\n",
    "| requests + base64 | Logo fetching and embedding |\n",
    "| pathlib + re | Batch file discovery and filename parsing |\n",
    "| Google Colab | Development environment |\n",
    "\n",
    "---\n",
    "\n",
    "## How to Run\n",
    "\n",
    "1. Upload the match CSVs and logos CSV to Google Drive\n",
    "2. Open the notebook in Google Colab\n",
    "3. Update the three paths in **Cell 2**\n",
    "4. Run all cells — processing ~380 files takes roughly 3-5 minutes\n",
    "5. Open the exported HTML in any browser\n",
    "\n",
    "> **Note:** The 380 match CSVs are not included due to file size. Source data available from [StatsBomb Open Data](https://github.com/statsbomb/open-data).\n",
    "\n",
    "---\n",
    "\n",
    "## About\n",
    "\n",
    "Built as part of a data analytics portfolio during an MSc in Applied Performance Analysis at the **University of Essex**.\n",
    "\n",
    "The project demonstrates end-to-end data work: raw event ingestion, domain-specific logic (goal attribution, tiebreak rules), iterative visual design, and front-end polish — all within a single reproducible notebook.\n",
    "\n",
    "---\n",
    "\n",
    "*Data © StatsBomb. Club badges © Premier League. Project code MIT licensed.*\n",
]

with open("README.md", "w", encoding="utf-8") as f:
    f.writelines(lines)

!git add README.md
!git commit -m "Complete README with live demo link and full documentation"
!git push origin main

[main 8ad1a9e] Complete README with live demo link and full documentation
 1 file changed, 93 insertions(+)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 2.27 KiB | 2.27 MiB/s, done.
Total 3 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/LeiderIP/Premier-League-2024-25-Title-race.git
   5023d05..8ad1a9e  main -> main
